#Bronze to Silver Transformation
**Project:** NYC TLC Azure Data Platform  
**Layer:** Silver  
**Description:** This notebook reads raw Parquet files from the Bronze layer, cleans and enriches the data, and writes Delta tables to the Silver layer.  
**Author:** Saikumar
**Created:** 2026


In [0]:
# Storage configuration
STORAGE_ACCOUNT = "nycrawdatazone"
CONTAINER       = "nyc-tlc"
ADLS_ENDPOINT   = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Layer paths
BRONZE_PATH     = f"{ADLS_ENDPOINT}/raw"
SILVER_PATH     = f"{ADLS_ENDPOINT}/silver"
LOOKUP_PATH     = f"{ADLS_ENDPOINT}/lookup"

# Year
YEARS            = ["2024","2025"]

# Taxi types
TAXI_TYPES      = ["yellow", "green", "fhv", "fhvhv"]

print(f"Bronze path : {BRONZE_PATH}")
print(f"Silver path : {SILVER_PATH}")
print(f"Lookup path : {LOOKUP_PATH}")

In [0]:
#Spark Optimizations
spark.conf.set("spark.sql.shuffle.partitions", "8")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

## Load Look-up Table

In [0]:
#Read taxi zone look up table

taxi_zones=spark.read.option("header","true").option("inferSchema","true")\
            .format("csv").load(f"{LOOKUP_PATH}/taxi_zone_lookup.csv")

#Caching taxi zone table
taxi_zones.cache()
taxi_zones.count()
taxi_zones.printSchema()
taxi_zones.show(5)

## Process Yellow Taxi

In [0]:
yellow_taxi= spark.read.option("header","true").option("inferSchema","true").parquet(f"{BRONZE_PATH}/yellow/year=2024/month=01/*")
display(yellow_taxi.limit(5))

display(yellow_taxi.select('passenger_count').distinct())

In [0]:
from pyspark.sql.functions import col, lit, when, year, month, unix_timestamp, to_date, broadcast

def process_yellow_taxi(yr):

  df=spark.read.option("header","true").option("inferSchema","true").parquet(f"{BRONZE_PATH}/yellow/year={yr}/*/*")

  print("Row row count: ",df.count())
  print("Raw column count: ", len(df.columns))

  #Dropping duplicates and filtering
  df= df.dropDuplicates()

  df= df.dropna(subset=['tpep_pickup_datetime','tpep_dropoff_datetime','PULocationID','DOLocationID'])

  df= df .filter(col("trip_distance")>0)

  df= df.filter(col("fare_amount")>0)

  df= df.filter((col("passenger_count")>0) & (col("passenger_count")<=8))

  df= df.filter((col("tpep_pickup_datetime")<col("tpep_dropoff_datetime")))

  df= df.filter(
    (col("tpep_pickup_datetime") >= f"{yr}-01-01") &
    (col("tpep_pickup_datetime") <= f"{yr}-12-31")
    )
  
  #Adding new Columns

  df= df.withColumn("trip_duration_mins", (unix_timestamp(col("tpep_dropoff_datetime")) - 
                                           unix_timestamp(col("tpep_pickup_datetime")))/60)

  df= df.withColumn("pickup_date", to_date(col("tpep_pickup_datetime")))

  df= df.withColumn("taxi_type", lit("yellow"))

  if "cbd_congestion_fee" not in df.columns:
    df= df.withColumn("cbd_congestion_fee", lit(0.0))
  else:
    df= df.withColumn("cbd_congestion_fee",when(col("cbd_congestion_fee").isNull(), 0.0)\
        .otherwise(col("cbd_congestion_fee")))
  
  #Joining with taxi zone lookup
  df= df.join(broadcast(taxi_zones.select(
    col("LocationID").alias("PULocationID"),
    col("Zone").alias("pickup_zone"),
    col("Borough").alias("pickup_borough")
  )), on="PULocationID", how="left")

  df= df.join(broadcast(taxi_zones.select(
    col("LocationID").alias("DOLocationID"),
    col("Zone").alias("dropoff_zone"),
    col("Borough").alias("dropoff_borough")
  )), on="DOLocationID", how="left")

  df= df.select(
    "taxi_type",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "trip_duration_mins",
    "pickup_date",
    "PULocationID",
    "DOLocationID",
    "pickup_borough",
    "pickup_zone",
    "dropoff_borough",
    "dropoff_zone",
    "payment_type",
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "congestion_surcharge",
    "airport_fee",
    "cbd_congestion_fee",
    "total_amount"
  )

  print("Cleaned row count: ",df.count())
  print("Cleaned column count: ", len(df.columns))

  display(df.limit(10))

  df.write.mode("overwrite").format("delta").partitionBy("pickup_date")\
  .option("mergeSchema","true")\
  .option("replaceWhere",
          f"pickup_date>= '{yr}-01-01' AND pickup_date <= '{yr}-12-31'").save(f"{SILVER_PATH}/yellow")
  
for yr in YEARS:
  process_yellow_taxi(yr)  

#Process Green Taxi

In [0]:
green_taxi=spark.read.option("header","true").option("inferSchema","true").parquet(f"{BRONZE_PATH}/green/year=2024/month=01/*")

display(green_taxi.limit(5))



In [0]:
from pyspark.sql.functions import date_format

def process_green_taxi(yr):

    df= spark.read.option("header","true").option("inferSchema","true")\
        .parquet(f"{BRONZE_PATH}/green/year={yr}/*/*")

    print("Raw row count: ",df.count())
    print("Raw column count: ", len(df.columns))

    #Clean data

    df= df.dropDuplicates()

    df= df.dropna(subset=['lpep_pickup_datetime','lpep_dropoff_datetime','PULocationID','DOLocationID'])

    df= df.filter(col("trip_distance")>0)

    df= df.filter(col("fare_amount")>0)

    df= df.filter((col("passenger_count")>0) & (col("passenger_count")<=8))

    df= df.filter((col("lpep_pickup_datetime")<col("lpep_dropoff_datetime")))

    df= df.filter(
      (col("lpep_pickup_datetime") >= f"{yr}-01-01") &
      (col("lpep_pickup_datetime") <= f"{yr}-12-31")
      )

    #Adding new columns

    df= df.withColumn("trip_duration_mins", (unix_timestamp(col("lpep_dropoff_datetime")) - 
                                             unix_timestamp(col("lpep_pickup_datetime")))/60)

    df= df.withColumn("pickup_date", to_date(col("lpep_pickup_datetime")))

    df= df.withColumn("pickup_month", date_format(col("lpep_pickup_datetime"),"yyyy-MM"))

    df= df.withColumn("taxi_type", lit("green"))

    if "cbd_congestion_fee" not in df.columns:
        df= df.withColumn("cbd_congestion_fee", lit(0.0))
    else:
        df= df.withColumn("cbd_congestion_fee",when(col("cbd_congestion_fee").isNull(), 0.0)\
            .otherwise(col("cbd_congestion_fee")))

    #Joining with taxi zone lookup
    df= df.join(broadcast(taxi_zones.select(
    col("LocationID").alias("PULocationID"),
    col("Zone").alias("pickup_zone"),
    col("Borough").alias("pickup_borough")
     )), on="PULocationID", how="left")

    df= df.join(broadcast(taxi_zones.select(
    col("LocationID").alias("DOLocationID"),
    col("Zone").alias("dropoff_zone"),
    col("Borough").alias("dropoff_borough")
     )), on="DOLocationID", how="left")
    
    df= df.select(
        "taxi_type",
        "lpep_pickup_datetime",
        "lpep_dropoff_datetime",
        "trip_duration_mins",
        "pickup_date",
        "pickup_month",
        "PULocationID",
        "DOLocationID",
        "pickup_borough",
        "pickup_zone",
        "dropoff_borough",
        "dropoff_zone",
        "payment_type",
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "tolls_amount",
        "total_amount",
        "congestion_surcharge",
        "cbd_congestion_fee"
    )

    print("Cleaned row count: ",df.count())
    print("Cleaned column count: ", len(df.columns))

    display(df.limit(10))

    df.write.mode("overwrite").format("delta").partitionBy("pickup_month")\
    .option("mergeSchema","true")\
    .option("replaceWhere", f"pickup_month>='{yr}-01' AND pickup_month<='{yr}-12'")\
    .save(f"{SILVER_PATH}/green")
  
for yr in YEARS:
    process_green_taxi(yr)

#Process FHV

In [0]:
for month in range(1,13):
    if month<10:
        month=f"0{month}"
    fhv=spark.read.option("header","true").option("inferSchema","true").parquet(f"{BRONZE_PATH}/fhv/year=2024/month={month}/*")
    display(fhv.printSchema())


In [0]:
def process_fhv_taxi(yr):

    df= spark.read.option("inferSchema","true").option("header","true").parquet(f"{BRONZE_PATH}/fhv/year={yr}/*/*")

    print("Raw row count: ",df.count())
    print("Raw column count: ", len(df.columns))

    #Clean data

    df= df.dropDuplicates()

    df= df.dropna(subset=['pickup_datetime','dropOff_datetime','PUlocationID','DOlocationID'])

    df= df.filter(col("dropOff_datetime")>col("pickup_datetime"))

    df= df.filter(
        (col("pickup_datetime") >= f"{yr}-01-01") &
        (col("pickup_datetime") <= f"{yr}-12-31")
    )

    #Adding new columns

    df= df.withColumn("trip_duration_mins", (unix_timestamp(col("dropOff_datetime")) - 
                                             unix_timestamp(col("pickup_datetime")))/60)

    df= df.withColumn("pickup_date", to_date(col("pickup_datetime")))

    df= df.withColumn("pickup_month", date_format(col("pickup_datetime"),"yyyy-MM"))

    df= df.withColumn("taxi_type", lit("fhv"))

    #Change type of location IDs to match taxi_zones LocationID type
    df = df.withColumn("PUlocationID", col("PUlocationID").cast("integer"))
    df = df.withColumn("DOlocationID", col("DOlocationID").cast("integer"))

    #Joining with taxi zone lookup
    df= df.join(broadcast(taxi_zones.select(
    col("LocationID").alias("PUlocationID"),
    col("Zone").alias("pickup_zone"),
    col("Borough").alias("pickup_borough")
     )), on="PUlocationID", how="left")

    df= df.join(broadcast(taxi_zones.select(
    col("LocationID").alias("DOlocationID"),
    col("Zone").alias("dropoff_zone"),
    col("Borough").alias("dropoff_borough")
     )), on="DOlocationID", how="left")
    
    df= df.select(
        "taxi_type",
        "pickup_datetime",
        "dropOff_datetime",
        "trip_duration_mins",
        "pickup_date",
        "pickup_month",
        "PUlocationID",
        "DOlocationID",
        "pickup_borough",
        "pickup_zone",
        "dropoff_borough",
        "dropoff_zone"
        )
    print("Cleaned row count: ", df.count())
    print("Cleaned column count: ", len(df.columns))

    display(df.limit(10))

    df.write.mode("overwrite").format("delta").partitionBy("pickup_month")\
    .option("mergeSchema","true")\
    .option("replaceWhere", f"pickup_month >= '{yr}-01' AND pickup_month <= '{yr}-12'")\
    .save(f"{SILVER_PATH}/fhv")
  
for yr in YEARS:
    process_fhv_taxi(yr)

#Process HVFHV

In [0]:
fhvhv=spark.read.option("header","true").option("inferSchema","true").parquet(f"{BRONZE_PATH}/fhvhv/year=2025/month=*/*")
display(fhvhv.printSchema())
display(fhvhv.limit(5))

print(fhvhv.count())

In [0]:
from pyspark.sql.functions import col, lit, when, year, month, unix_timestamp, to_date, broadcast, date_format

def process_fhvhv(yr):
    
    #Read Bronze
    df = spark.read \
        .option("mergeSchema", "true") \
        .parquet(f"{BRONZE_PATH}/fhvhv/year={yr}/*/*")
    
    print(f"Raw row count     : {df.count():,}")
    print(f"Raw column count  : {len(df.columns)}")
    
    #Clean
    df = df.dropDuplicates()

    df = df.dropna(subset=[
        "pickup_datetime",
        "dropoff_datetime",
        "PULocationID",
        "DOLocationID"])
    
    df = df.filter(col("trip_miles") > 0)

    df = df.filter(col("base_passenger_fare") > 0)

    df = df.filter(col("dropoff_datetime") > col("pickup_datetime"))

    df = df.filter(
        (col("pickup_datetime") >= f"{yr}-01-01") &
        (col("pickup_datetime") <= f"{yr}-12-31")
    )
    
    #Derived columns
    df = df.withColumn("trip_duration_mins",
        (unix_timestamp(col("dropoff_datetime")) -
         unix_timestamp(col("pickup_datetime"))) / 60
    )

    df = df.withColumn("pickup_date",
        to_date(col("pickup_datetime"))
    )

    df = df.withColumn("taxi_type", lit("fhvhv"))
    
    if "cbd_congestion_fee" not in df.columns:
        df = df.withColumn("cbd_congestion_fee", lit(0.0))
    else:
        df = df.withColumn("cbd_congestion_fee",
            when(col("cbd_congestion_fee").isNull(),0.0).otherwise(col("cbd_congestion_fee")))
    
    #Join with taxi zone lookup
    df = df.join(
        broadcast(taxi_zones.select(
            col("LocationID").alias("PULocationID"),
            col("Borough").alias("pickup_borough"),
            col("Zone").alias("pickup_zone")
        )),
        on="PULocationID",
        how="left"
    )
    df = df.join(
        broadcast(taxi_zones.select(
            col("LocationID").alias("DOLocationID"),
            col("Borough").alias("dropoff_borough"),
            col("Zone").alias("dropoff_zone")
        )),
        on="DOLocationID",
        how="left"
    )
    
    # Select relevant columns
    df = df.select(
        "taxi_type",
        "pickup_datetime",
        "dropoff_datetime",
        "trip_duration_mins",
        "pickup_date",
        "PULocationID",
        "DOLocationID",
        "pickup_borough",
        "pickup_zone",
        "dropoff_borough",
        "dropoff_zone",
        "trip_miles",
        "shared_request_flag",
        "shared_match_flag",
        "base_passenger_fare",
        "tolls",
        "congestion_surcharge",
        "airport_fee",
        "tips",
        "driver_pay",
        "cbd_congestion_fee"
    )
    
    print(f"Clean row count   : {df.count():,}")
    print(f"Clean column count: {len(df.columns)}")
    
    #Write to Silver
    df.write.format("delta").mode("overwrite").partitionBy("pickup_date") \
      .option("mergeSchema", "true")\
      .option("replaceWhere",f"pickup_date >= '{yr}-01-01' AND pickup_date <= '{yr}-12-31'") \
      .save(f"{SILVER_PATH}/fhvhv")

for yr in YEARS:
    process_fhvhv(yr)

In [0]:
# Verifying all Silver tables

print("Verifying Silver layer...\n")

for taxi in TAXI_TYPES:
    try:
        df = spark.read.format("delta").load(f"{SILVER_PATH}/{taxi}")
        print(f"{taxi:8} : {df.count():>15,} rows | {len(df.columns)} columns")
    except Exception as e:
        print(f"{taxi:8} : {str(e)[:100]}")

# remove taxi_zones cache
taxi_zones.unpersist()

print("\ntaxi_zones cache cleared")
print("\nSilver layer complete!")